In [1]:
import h5py
import awkward as ak
import os

In [2]:
data = ak.from_parquet("/local/norman/UHHAnalysis/SPANet/22pre_v14/hh_vbf_hbb_htt_kv1_k2v0_kl1_madgraph/nominal/calib__default/sel__default/red__default/prod24/events_0.parquet")

In [12]:
data.SPANetJet.fields

['btagDeepFlavB',
 'btagDeepFlavCvB',
 'btagDeepFlavCvL',
 'btagDeepFlavQG',
 'btagPNetB',
 'btagPNetCvB',
 'btagPNetCvL',
 'btagPNetCvNotB',
 'btagPNetQvG',
 'btagPNetTauVJet',
 'btagUParTAK4B',
 'btagUParTAK4CvB',
 'btagUParTAK4CvL',
 'btagUParTAK4CvNotB',
 'btagUParTAK4QvG',
 'btagUParTAK4TauVJet',
 'deterministic_seed',
 'eta',
 'hadronFlavour',
 'hhbtag',
 'mass',
 'phi',
 'pt']

In [2]:
file_path = '/home/norman/UHHAnalysis/SPANet/data/full_hadronic_ttbar/example.h5'

In [3]:
if os.path.exists(file_path):
    with h5py.File(file_path, 'r') as f:
        print("HDF5 Keys:", list(f.keys()))

HDF5 Keys: ['INPUTS', 'TARGETS']


In [4]:
with h5py.File(file_path, 'r') as f:
    # 1. List everything inside INPUTS
    print("Groups inside INPUTS:", list(f['INPUTS'].keys()))
    
    # 2. List everything inside TARGETS
    print("Groups inside TARGETS:", list(f['TARGETS'].keys()))
    
    # Let's try to find the first sequential input automatically
    first_input_name = list(f['INPUTS'].keys())[0]
    print(f"\nAccessing first input: '{first_input_name}'")
    
    input_group = f['INPUTS'][first_input_name]
    print(f"Features in {first_input_name}:", list(input_group.keys()))

Groups inside INPUTS: ['Source']
Groups inside TARGETS: ['t1', 't2']

Accessing first input: 'Source'
Features in Source: ['MASK', 'btag', 'eta', 'mass', 'phi', 'pt']


In [5]:
with h5py.File(file_path, 'r') as f:
    # Access the Source group
    source = f['INPUTS']['Source']
    
    # Get the data
    pt_data = source['pt'][:]
    mask_data = source['MASK'][:]
    
    print(f"Dataset has {pt_data.shape[0]} events.")
    print(f"Maximum jets per event: {pt_data.shape[1]}")
    
    print("\n--- First Event Data ---")
    print(f"PT values: {pt_data[0]}")
    print(f"Mask:      {mask_data[0]}")
    
    # Access Target data
    t1_q1 = f['TARGETS']['t1']['q1'][:]
    print(f"\nTarget t1/q1 (First 5 events): {t1_q1[:5]}")

Dataset has 10000 events.
Maximum jets per event: 10

--- First Event Data ---
PT values: [86.79656  68.08204  61.957756 61.809544 40.90059  32.217583 27.135532
 27.007011 22.368986  0.      ]
Mask:      [ True  True  True  True  True  True  True  True  True False]

Target t1/q1 (First 5 events): [ 1  0 -1  0  3]


In [6]:
def create_dataset(events, output_path):
    n_events = len(events)
    
    with h5py.File(output_path, 'w') as f:
        inputs = f.create_group('INPUTS')
        jets_group = inputs.create_group('Jets')  # This name must match your YAML
        
        # --- PREPARE INPUT FEATURES ---
        # We pad the jet collection to MAX_JETS
        padded_jets = ak.pad_none(events.Jet, MAX_JETS, clip=True)
        
        # Create the MASK (True if jet is real, False if padding)
        mask = ~ak.is_none(padded_jets.pt, axis=1)
        jets_group.create_dataset('MASK', data=mask.to_numpy())
        
        # Add features (pt, eta, phi, btag...)
        features = ['pt', 'eta', 'phi', 'btagDeepFlavB']
        for feat in features:
            # Fill padding (None) with 0.0
            data = ak.fill_none(padded_jets[feat], 0.0).to_numpy()
            jets_group.create_dataset(feat, data=data)
            
        # --- PREPARE TARGETS ---
        targets = f.create_group('TARGETS')
        
        # Example: Match HHBJet to the indices in the main Jet collection
        # We find the index in 'Jet' that matches the 'HHBJet'
        def get_indices(main_jets, target_sub_jets):
            # This is a simple matching logic based on pt/eta/phi or a seed
            # For now, let's assume we want the first two jets of HHBJet
            indices = []
            # ... matching logic here ...
            return indices

        # For now, let's create empty/dummy targets so you can see the structure
        # You will need to fill these with the real matched indices
        h_bb = targets.create_group('higgs_bb')
        h_bb.create_dataset('b1', data=np.full(n_events, -1, dtype=np.int64))
        h_bb.create_dataset('b2', data=np.full(n_events, -1, dtype=np.int64))
        
        vbf = targets.create_group('vbf')
        vbf.create_dataset('q1', data=np.full(n_events, -1, dtype=np.int64))
        vbf.create_dataset('q2', data=np.full(n_events, -1, dtype=np.int64))

    print(f"Created {output_path}")

In [7]:
create_dataset(events, 'vbf_hh_training.h5')

NameError: name 'events' is not defined

In [8]:
data = ak.from_parquet("/home/norman/UHHAnalysis/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v0_kl1_madgraph_events_0.parquet")

In [12]:
data.TrainingJet.fields

['btagDeepFlavB',
 'btagDeepFlavCvB',
 'btagDeepFlavCvL',
 'btagDeepFlavQG',
 'btagPNetB',
 'btagPNetCvB',
 'btagPNetCvL',
 'btagPNetCvNotB',
 'btagPNetQvG',
 'btagPNetTauVJet',
 'eta',
 'hadronFlavour',
 'hhbtag',
 'isVBF',
 'mass',
 'phi',
 'pt']